# 06 — Push Notifications: `tasks/pushNotificationConfig/set` and Webhooks

## Why this notebook exists

**Streaming** (notebook 05) is great while a connection stays open. **Polling** (notebook 04) works but burns requests. Both assume the client is *online and waiting* for the task to finish.

What if the client is a serverless function that gets killed after 30 seconds? A mobile app whose user backgrounded the screen? A CI job that fires-and-forgets and only cares about the eventual outcome?

A2A's third asynchrony pattern is **push notifications**. The client registers a webhook URL once via `tasks/pushNotificationConfig/set`, hangs up, and goes about its life. When the task reaches a terminal state, the server POSTs the final `Task` object to the webhook URL. No connection held, no polling cost.

This notebook stands up two FastAPI apps in one kernel — the researcher (port 8010) and a tiny callback receiver (port 8011) — and watches a task complete via webhook.

> *Targets A2A spec v0.3.0. Auth on push notifications is deferred to notebook 07.*

## What you'll learn

- The `tasks/pushNotificationConfig/set` method and its `TaskPushNotificationConfig` params.
- The shape of the **webhook payload** A2A servers POST when a task completes: just the serialized `Task` object, no envelope.
- How to register a webhook from the client, fire-and-forget the task, and pick up the result via callback.
- How to run a second FastAPI app **in the same notebook** to act as the webhook receiver.
- When to prefer push notifications over polling or streaming.
- Why webhook auth matters (and why we're deferring it one notebook).

## 1. Setup

Same helpers as previous notebooks.

In [ ]:
import json
import threading
import time
import uuid
from datetime import datetime, timezone
from typing import Literal

import httpx
import uvicorn
from fastapi import FastAPI, Request
from pydantic import BaseModel, Field, ValidationError, model_validator

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")
    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


print("Setup OK")

## 2. When SSE Isn't Enough

| Scenario | Best fit |
|---|---|
| Client wants every progress update in real time, will stay online | **Streaming** (`message/stream`, notebook 05) |
| Client wants to know when it's done, may go away in the meantime | **Push notifications** (this notebook) |
| Client controls its own cadence, infrastructure forbids long-lived connections | **Polling** (`tasks/get`, notebook 04) |

The webhook flow:

```
client ──────── message/send ───────► server   (returns submitted Task)
client ─── pushNotificationConfig/set ───► server   (stores webhook URL for task)
client closes the connection. Goes about its day.

…time passes, server's worker finishes the task…

server ──────── POST final Task ─────► client's webhook
```

The webhook payload is just the serialized `Task` object — no JSON-RPC envelope, no event wrapper. That's a deliberate simplification: the receiver only needs to know how to parse a `Task`, not the whole A2A protocol.